In [1]:
from pathlib import Path

import yaml
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

d:\Translator\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path.cwd()
while not (ROOT / "configs" / "config.yaml").exists():
    ROOT = ROOT.parent

with open(ROOT / "configs" / "config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

vocab_size = config["tokenizer"]["vocab_size"]
special_tokens = list(config["tokenizer"]["special_tokens"].values())
print(vocab_size, special_tokens)

8000 ['<pad>', '<sos>', '<eos>', '<unk>']


In [3]:
ds = load_dataset(config["dataset"]["name"], split="train")
print(ds)

'[Errno 11002] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/bentrevett/multi30k/resolve/4589883f3d09d4ef6361784e03f0ead219836469/multi30k.py
Retrying in 1s [Retry 1/5].
'[Errno 11002] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/datasets/bentrevett/multi30k/resolve/4589883f3d09d4ef6361784e03f0ead219836469/multi30k.py
Retrying in 2s [Retry 2/5].


Dataset({
    features: ['en', 'de'],
    num_rows: 29000
})


In [4]:
data_dir = ROOT / "data"
data_dir.mkdir(exist_ok=True)
corpus_path = data_dir / "combined_corpus.txt"

with open(corpus_path, "w", encoding="utf-8") as f:
    for example in ds:
        f.write(example["de"] + "\n")
        f.write(example["en"] + "\n")

print(f"Wrote {corpus_path}")

Wrote d:\Translator\data\combined_corpus.txt


In [5]:
tokenizer = Tokenizer(BPE(unk_token=config["tokenizer"]["special_tokens"]["unk"]))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(vocab_size=vocab_size, special_tokens=special_tokens)
tokenizer.train([str(corpus_path)], trainer)

print(f"Trained vocab size: {tokenizer.get_vocab_size()}")


# Think of BpeTrainer like a custom dictionary builder:
# You give it a blank notebook with vocab_size number of pages.
# You pre-fill the first few pages with special_tokens (like emergency instructions).
# Then, when you run the trainer on your text, it fills the remaining pages with the most common word patterns it finds!

Trained vocab size: 8000


In [6]:
tokenizer_path = data_dir / "bpe_tokenizer.json"
tokenizer.save(str(tokenizer_path))
print(f"Saved tokenizer to {tokenizer_path}")

Saved tokenizer to d:\Translator\data\bpe_tokenizer.json


In [7]:
sample_de = ds[0]["de"]
sample_en = ds[0]["en"]

encoded_de = tokenizer.encode(sample_de)
encoded_en = tokenizer.encode(sample_en)

print(sample_de)
print(encoded_de.tokens)
print()
print(sample_en)
print(encoded_en.tokens)

Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.
['Zwei', 'junge', 'weiße', 'Männer', 'sind', 'im', 'Freien', 'in', 'der', 'Nähe', 'viel', 'er', 'Bü', 'sche', '.']

Two young, White males are outside near many bushes.
['Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.']
